***

* [总目录](../0_Introduction/0_introduction.ipynb)
* [术语表](../0_Introduction/1_glossary.ipynb)
* [第 4 章：可见度空间](4_0_introduction.ipynb)
    * 上一节：[4.5.1 $uv$ 轨迹跟踪](4_5_1_uv_coverage_uv_tracks.ipynb)
    * 下一节：[4.6 傅里叶近似与 van Cittert-Zernike 定理](4_6_the_fourier_approximation_van_cittert-zernike_theorem.ipynb)

***


导入标准模块。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
try:
    from IPython.display import HTML
except ImportError:
    def HTML(*args, **kwargs):
        return None
%matplotlib inline
HTML('../style/course.css')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False


In [ ]:
try:
    from IPython.display import HTML
except ImportError:
    def HTML(*args, **kwargs):
        return None
HTML('../style/code_toggle.html')


## 4.5.2 提高 $uv$ 覆盖<a id='visibility:sec:improving_uv_coverage'></a>

第 4.5.1 节只讨论了一条基线。真实干涉阵由多面天线组成，每一对天线都会形成一条基线。若阵列有 $N$ 面天线，同一时间、同一频率、同一极化下的独立互相关数量为

$$
N_{\rm bl}=\frac{N(N-1)}{2}.
$$

因此，增加天线数会快速增加瞬时 `uv` 采样点。但 `uv` 覆盖不仅由天线数决定，还取决于阵列布局、观测时长、源赤纬、频率覆盖、数据标记和权重。成像质量的许多差异，都可以追溯到这些因素如何塑造采样函数。


In [ ]:
fig_dir = Path('figures')
fig_dir.mkdir(exist_ok=True)


def make_demo_array():
    core = np.array([[0, 0], [35, 10], [-28, 22], [12, -42], [-48, -18], [58, -35]])
    arms = []
    for angle_deg in [15, 135, 255]:
        angle = np.deg2rad(angle_deg)
        for radius in [110, 230, 390, 620]:
            arms.append([radius*np.cos(angle), radius*np.sin(angle)])
    return np.vstack([core, np.array(arms)])


def baselines_enu(ant_xy):
    pairs = []
    n = len(ant_xy)
    for i in range(n):
        for j in range(i+1, n):
            east = ant_xy[j,0] - ant_xy[i,0]
            north = ant_xy[j,1] - ant_xy[i,1]
            pairs.append([east, north, 0.0])
    return np.array(pairs)


def enu_to_xyz(b_enu, lat_rad):
    E = b_enu[:,0]
    N = b_enu[:,1]
    U = b_enu[:,2]
    X = -np.sin(lat_rad)*N + np.cos(lat_rad)*U
    Y = E
    Z = np.cos(lat_rad)*N + np.sin(lat_rad)*U
    return np.vstack([X, Y, Z]).T


def uvw_from_xyz_many(b_xyz_m, hour_angle_rad, dec_rad, wavelength_m):
    b = np.asarray(b_xyz_m) / wavelength_m
    X, Y, Z = b[:,0], b[:,1], b[:,2]
    H = np.asarray(hour_angle_rad)
    all_u = []
    all_v = []
    for h in H:
        u = X*np.sin(h) + Y*np.cos(h)
        v = -X*np.sin(dec_rad)*np.cos(h) + Y*np.sin(dec_rad)*np.sin(h) + Z*np.cos(dec_rad)
        all_u.append(u)
        all_v.append(v)
    u = np.concatenate(all_u)
    v = np.concatenate(all_v)
    return np.concatenate([u, -u]), np.concatenate([v, -v])


def collect_uv(b_xyz_m, ha_h, dec_rad, wavelengths):
    all_u, all_v = [], []
    for lam in np.atleast_1d(wavelengths):
        u, v = uvw_from_xyz_many(b_xyz_m, np.asarray(ha_h)*np.pi/12.0, dec_rad, lam)
        all_u.append(u)
        all_v.append(v)
    return np.concatenate(all_u), np.concatenate(all_v)


def dirty_beam_from_uv(u, v, grid=192, uvmax=None):
    if uvmax is None:
        uvmax = max(np.max(np.abs(u)), np.max(np.abs(v))) * 1.05
    sampling = np.zeros((grid, grid), dtype=float)
    ui = np.round((u/uvmax + 1) * 0.5 * (grid-1)).astype(int)
    vi = np.round((v/uvmax + 1) * 0.5 * (grid-1)).astype(int)
    good = (ui >= 0) & (ui < grid) & (vi >= 0) & (vi < grid)
    sampling[vi[good], ui[good]] = 1.0
    beam = np.fft.fftshift(np.fft.ifft2(np.fft.ifftshift(sampling))).real
    beam /= np.max(np.abs(beam))
    return sampling, beam

ant = make_demo_array()
b_enu = baselines_enu(ant)
b_xyz = enu_to_xyz(b_enu, np.deg2rad(-30.7))
dec = np.deg2rad(-30.0)
lam0 = 0.21

snapshot_u, snapshot_v = collect_uv(b_xyz, [0.0], dec, lam0)
track4_u, track4_v = collect_uv(b_xyz, np.linspace(-2, 2, 25), dec, lam0)
track8_u, track8_v = collect_uv(b_xyz, np.linspace(-4, 4, 49), dec, lam0)
lam_band = np.linspace(0.18, 0.24, 7)
mfs_u, mfs_v = collect_uv(b_xyz, np.linspace(-4, 4, 49), dec, lam_band)

# Figure 4.5.4: array layout and snapshot coverage.
fig, axes = plt.subplots(1, 2, figsize=(12, 5.2), constrained_layout=True)
axes[0].scatter(ant[:,0], ant[:,1], s=42, color='#2b6cb0')
axes[0].set_aspect('equal', adjustable='box')
axes[0].set_xlabel('East [m]')
axes[0].set_ylabel('North [m]')
axes[0].set_title('Example antenna layout')
axes[0].grid(alpha=0.25)
axes[1].scatter(snapshot_u, snapshot_v, s=8, color='#c53030', alpha=0.75)
axes[1].set_aspect('equal', adjustable='box')
axes[1].set_xlabel('u [wavelengths]')
axes[1].set_ylabel('v [wavelengths]')
axes[1].set_title('Snapshot uv coverage')
axes[1].grid(alpha=0.25)
fig.savefig(fig_dir / 'uv_coverage_array_snapshot.png', dpi=180)
plt.close(fig)

# Figure 4.5.5: coverage and dirty beam improvements.
cases = [
    ('snapshot', snapshot_u, snapshot_v),
    ('4 h rotation', track4_u, track4_v),
    ('8 h rotation', track8_u, track8_v),
    ('8 h + multi-frequency', mfs_u, mfs_v),
]
uvmax = max(np.max(np.abs(mfs_u)), np.max(np.abs(mfs_v))) * 1.03
fig, axes = plt.subplots(2, 4, figsize=(15.5, 7.4), constrained_layout=True)
for col, (title, uu, vv) in enumerate(cases):
    sampling, beam = dirty_beam_from_uv(uu, vv, grid=192, uvmax=uvmax)
    axes[0, col].scatter(uu, vv, s=1.5, color='#2b6cb0', alpha=0.45)
    axes[0, col].set_xlim(-uvmax, uvmax)
    axes[0, col].set_ylim(-uvmax, uvmax)
    axes[0, col].set_aspect('equal', adjustable='box')
    axes[0, col].set_title(title)
    axes[0, col].set_xticks([])
    axes[0, col].set_yticks([])
    extent = [-1, 1, -1, 1]
    axes[1, col].imshow(beam, origin='lower', extent=extent, cmap='RdBu_r', vmin=-0.35, vmax=1.0)
    axes[1, col].set_xticks([])
    axes[1, col].set_yticks([])
    if col == 0:
        axes[0, col].set_ylabel('uv samples')
        axes[1, col].set_ylabel('dirty beam')
fig.savefig(fig_dir / 'uv_coverage_improvement_dirty_beam.png', dpi=180)
plt.close(fig)


### 4.5.2.1 天线数和阵列布局<a id='visibility:sec:antenna_layout'></a>

多天线阵列的第一种优势来自基线数。$N$ 面天线产生 $N(N-1)/2$ 条独立基线，每条基线还对应一个共轭采样点。因此，即使在快照观测中，一个多天线阵列也能在 `uv` 平面上留下许多点。

不过，基线数并不是全部。若天线布局过于规则，`uv` 点会集中在少数轨迹或半径上，脏波束旁瓣可能很强；若布局兼顾短、中、长基线，采样会同时覆盖大尺度结构和小尺度结构。紧凑核心提供短基线灵敏度，长臂提供角分辨率，二者之间的折中决定阵列适合的科学目标。


![天线布局与快照 uv 覆盖](figures/uv_coverage_array_snapshot.png)

**图 4.5.4** 示例阵列布局及其快照 `uv` 覆盖。每一对天线给出一条基线，因此天线布局会直接映射为 `uv` 平面的瞬时采样结构。


### 4.5.2.2 地球自转综合<a id='visibility:sec:earth_rotation_synthesis'></a>

第二种改善来自时间。随着地球自转，每条物理基线在相位中心切平面上的投影不断变化，于是每个快照采样点会被拉成一段轨迹。只要相位中心能够持续跟踪、数据校准稳定、源结构在观测期间不变，就可以把不同时间的样本视为同一个可见度函数上的不同采样点。

这就是地球自转综合。它不需要移动天线，而是利用地球自转改变投影几何。观测时间越长，轨迹覆盖越完整；但实际观测还受最低仰角、日夜调度、源可见时间、大气稳定性和阵列队列安排限制。


### 4.5.2.3 多频率综合<a id='visibility:sec:multi_frequency_synthesis'></a>

第三种改善来自频率。`u,v,w` 坐标以波长为单位，因此同一条物理基线在不同频率下对应不同的空间频率半径：

$$
(u,v,w)\propto \frac{1}{\lambda}=\frac{\nu}{c}.
$$

宽带观测会把每条轨迹沿径向方向展开，形成更厚的采样带，这称为多频率综合。它可以显著改善 `uv` 覆盖，但也引入新的物理问题：天空亮度随频率变化，主波束随频率变化，校准误差也可能具有频率结构。因此，多频率综合成像通常需要同时拟合强度和谱指数，不能简单把所有频率无差别堆在一起。


### 4.5.2.4 覆盖与脏波束<a id='visibility:sec:coverage_dirty_beam'></a>

`uv` 覆盖的直接图像域后果是脏波束。若采样函数为 $S(u,v)$，则脏波束近似为

$$
B_D(l,m)=\mathcal{F}^{-1}\{S(u,v)\}.
$$

稀疏、成束或高度规则的采样会产生强旁瓣；更均匀、更丰富的采样通常会降低旁瓣，使脏图像更容易解释，也让后续 CLEAN 更稳定。图 4.5.5 把快照、地球自转综合和多频率综合并排比较，展示采样函数如何直接改变脏波束。


![uv 覆盖与脏波束改善](figures/uv_coverage_improvement_dirty_beam.png)

**图 4.5.5** `uv` 覆盖改善策略及其对应脏波束。观测时间变长会把点扩展成轨迹，多频率综合会沿径向填充采样；采样越丰富，脏波束旁瓣通常越低。


### 4.5.2.5 权重、标记与实际取舍<a id='visibility:sec:coverage_practical_tradeoffs'></a>

实际成像中，`uv` 覆盖并不只由观测设计决定。坏数据标记会挖掉部分采样；不同天线灵敏度和系统温度会改变样本权重；自然加权、均匀加权和 Briggs 鲁棒加权会重新塑造有效采样函数。因而“覆盖更多”并不总是唯一目标，还要同时考虑噪声、分辨率、旁瓣和对大尺度结构的敏感性。

一个实用判断是：短基线决定大尺度结构能否被恢复，长基线决定角分辨率，覆盖的方位角分布决定脏波束是否具有强方向性旁瓣。观测设计和成像加权本质上都在调节这三者之间的关系。


### 4.5.2.6 本节要点

提高 `uv` 覆盖有三条主要路径：增加天线数和优化阵列布局，利用地球自转综合延长轨迹，利用宽带观测进行多频率综合。`uv` 覆盖决定采样函数，采样函数的反傅里叶变换决定脏波束。好的覆盖并不只是点数多，还要在空间频率半径、方位角和权重分布上适合科学目标。


***

* 下一节：[4.6 傅里叶近似与 van Cittert-Zernike 定理](4_6_the_fourier_approximation_van_cittert-zernike_theorem.ipynb)
